<a href="https://colab.research.google.com/github/Song-yiJung/korean-modern-document-ocr/blob/main/02_step2_gemini/step2_gemini_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Step 2: Gemini 멀티모달 교정**


* 매뉴얼 챕터 3.2 대응
* Step 1의 결과(vision_raw + 원본 이미지)를 입력으로 받아
도메인 특화 프롬프트(SYSTEM_PROMPT)와 함께 Gemini에 보내 교정한다.

*  Step 1의 Vision OCR이 *형태 인식*만 한 결과를, **문맥을 보는** Gemini가
원본 이미지와 함께 교차 검증해 재판독·정리한다. 결과는 step1이 만든
같은 JSON 파일의 `gemini_corrected` 필드에 채워지고, 별도의
`<파일ID>_gemini_final.txt`도 함께 생성된다.

- *본인이 직접 수정해야 할 부분은 두 셀*:

   ③ ★ 사용자 설정   — 키 이름·경로·모델·간격

   ⑤ ★ SYSTEM_PROMPT — 본인 사료 도메인에 맞게 (기본값 그대로 사용도 가능)

*준비*
1. Step 1이 완료되어 `OUTPUT_DIR` 폴더에 JSON들이 있어야 함
2. Gemini API 키 (Google AI Studio에서 발급 — 부록 A.3)
3. 본인 사료에 맞는 SYSTEM_PROMPT (셀 5 — *반드시* 재작성)

## **시작 전 준비**

### 1. 이 노트북을 본인 계정에 복사

상단 메뉴에서 **`파일 → Drive에 사본 저장`** 을 클릭한다.
원본은 읽기 전용이므로 *반드시* 본인 계정의 사본으로 작업한다.
복사된 노트북은 본인 Google Drive의 `Colab Notebooks` 폴더에
자동 저장된다.

### 2. Step 1 결과가 준비되어 있어야 함

`step1_vision_colab.py`를 먼저 실행해서 `OUTPUT_DIR` 폴더에
사료별 JSON이 생성된 상태여야 한다.

### 3. Gemini API 키 발급

부록 A.3 절차로 Google AI Studio에서 발급. 발급된 키 문자열
(`AIzaSy...`로 시작)은 한 번만 보이므로 즉시 메모해 둔다.

### 4. ★ 두 노트북의 폴더 경로는 동일해야 함 ★

step2의 `INPUT_DIR`(원본 이미지)과 `OUTPUT_DIR`(step1 결과)는
step1에서 사용한 값과 *정확히 같아야* 한다. 다르면 step2가
step1 결과를 찾지 못한다.

---

#### 예상 시간·비용 (참고)

- 처리 시간: 사료 한 장당 약 30초 (Flash 기준)
- 비용: 장당 약 한화 1 ~ 2원 (Flash, 1,000자 내외 사료 기준)
  100장 처리 시 한화 100~200원, 무료 등급 분당 한도(10 RPM) 내 작업 가능

## 1) 필요한 패키지 설치

* 코랩은 매 세션마다 환경이 초기화되므로 *매번* 실행한다.


In [ ]:
!pip install -q google-generativeai pillow

## ② Google Drive 마운트

Step 1과 같은 세션이면 이미 마운트되어 있을 수 있다. 그래도 다시
실행해도 무방하다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


# %% [markdown]

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 사용자 설정 (5개)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# [1] Gemini API 키를 Colab Secrets에 등록할 때 사용한 이름.
#     좌측 🔑 → 새 보안 비밀 → 이름란에 입력했던 그 이름.
#     값에는 AI Studio에서 발급받은 `AIzaSy...`로 시작하는 문자열.
GEMINI_KEY_SECRET_NAME = 'GEMINI_API_KEY'

# [2] 입력·출력 폴더 (보통 동일 — Step 1의 결과 폴더 그대로).
#     Step 1과 같은 폴더로 두면 같은 JSON 파일의 `gemini_corrected` 필드를 채운다.
INPUT_DIR  = '/content/drive/MyDrive/사료_이미지'   # 원본 이미지 (vision_raw에 저장된 file_path를 해소)
OUTPUT_DIR = '/content/drive/MyDrive/vision_결과'   # Step 1이 저장한 JSON들 (여기에 교정 결과도 추가)

# [3] Gemini 모델.
#     - 'models/gemini-2.5-flash'      — 빠르고 저렴 (권장)
#     - 'models/gemini-2.5-pro'        — 정확하지만 느리고 비쌈
#     - 'models/gemini-2.5-flash-lite' — 가장 저렴, 정확도 살짝 낮음
MODEL_NAME = 'models/gemini-2.5-flash'

# [4] 호출 간격(초). 분당 호출 한도(RPM)에 맞춰 자동 산출.
#     무료 등급 Flash는 분당 10회. 유료는 RPM ↑.
SLEEP_INTERVAL_SEC = 6

# [5] 환율 (참고용 — 청구액 한화 환산).
USD_TO_KRW = 1400

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## 4) Gemini API 인증

In [ ]:
import google.generativeai as genai
from google.colab import userdata

genai.configure(api_key=userdata.get(GEMINI_KEY_SECRET_NAME))
model = genai.GenerativeModel(MODEL_NAME)
print(f"모델: {MODEL_NAME}")
print("Gemini 인증 완료.")

## 5) ★ SYSTEM_PROMPT — 본인 사료 도메인에 맞게 재작성

**여기가 본 노트북에서 *가장 중요한 부분*이다.**
프롬프트의 *구체적 도메인 어휘·규칙*이 결과의 질을 결정한다.

아래는 **1930년대 조선총독부 행정문서 + 불국사 사료** 도메인용 *예시* 프롬프트(v3).
본인 사료(조선시대 한문 문서, 근대 한글 신문, 서양 라틴어 사료 등)에 적용하려면
**블록 ①~⑤를 본인 도메인의 내용으로 채워야 한다**:

- **블록 ①** 도입부: 사료의 시대·언어·매체·특성
- **블록 ②** 구조 보존: 표·세로쓰기 등 본 사료의 구조 규칙
- **블록 ③** 언어·문자 정밀도: 본 사료에서 자주 혼동되는 글자·표기
- **블록 ④** 도메인 어휘: 본 사료에 자주 등장하는 고유명사·인명·지명·전문 용어
- **블록 ⑤** 환각 방지·출력 통제: 추측 금지, 표기 일관성

본 프롬프트는 약 3,000자. 본인 도메인용은 길이 *비슷하게* 유지 권장
(너무 길면 attention 분산, 너무 짧으면 도메인 단서 부족).

In [ ]:
SYSTEM_PROMPT = """당신은 1930년대 일제강점기(1910~1945) 조선총독부 행정문서부터 민간 서신, 전보에 이르는 문화유산 기록물 해독에 통달한 역사학자이자 아키비스트이다.
본 사료는 '불국사(佛國寺)' 및 관련 문화재를 대상으로 한 공문서, 서신, 전보 등으로 일본 한자(구자체), 히라가나, 가타카나, 숫자가 혼용되어 있으며, 인쇄체와 수기체(필기)가 섞여 있다.

첨부된 원본 이미지와 1차 Vision API가 물리적으로 추출한 원시 텍스트(vision_raw)를 교차 검증하여, 다음 학술적 지침에 따라 텍스트를 교정 및 복원한다.

[교정 및 복원 지침]

1. 표·목록 레이아웃 보존 (Tabular Layout Preservation) — 최우선 규칙:
   - 동일 열의 값이 반복될 때는 절대 평탄화하지 말고 반드시 '〃' 또는 '全' 기호로 표기한다.
   - 행정 계층(郡/面/里)은 각 행에서 동일 상위가 반복되면 '〃'로 축약하되, 상위 계층을 임의로 생략하거나 압축하지 말 것.
   - 열 간 공백은 이미지에 보이는 열 구분을 반영하여 탭/다수 공백으로 유지한다.

2. 일본어 법령체 접속사·조사 정밀 판독:
   - '又ハ'(또는)과 '及'(그리고)를 절대 혼동하지 말 것.
   - 법령·공문서 상투어는 시대 용례를 반영한다 (예: '保存上必要ト認メラルル事項', 'ニ關スル', 'ニ付').
   - 조동사 활용('タル', 'ベキ', 'セシ', 'ラル')을 임의로 단순화하지 말 것.

3. 표준 조사서 템플릿 인식:
   고적·보물 조사서는 다음 8개 고정 필드를 따른다. 번호를 중복하거나 건너뛰지 말 것.
     一、名稱   二、所在地   三、地域、地番、地目及地積   四、工作物其ノ他ノ物件ノ名稱、員數、品質、形狀、構造、形式及大サ
     五、現狀   六、由來又ハ傳說   七、保存上必要ト認メラルル事項   八、其他參考ト爲ルベキ事項

4. 숫자·도량형 정확성:
   - 자릿수 구분 쉼표/점을 반드시 보존한다 (예: '三,三〇九坪'을 '三九坪'으로 축약 금지).
   - 치수 단위(尺·寸·間·段·坪)는 이미지 판독에 근거; 추측 금지.
   - 연호는 '昭和N年M月D日' 형식으로 풀어쓴다.

5. 한자 변이자(구자체) 보존:
   - 현대 일본 상용한자(신자체)로 바꾸지 말 것:
     臺(O)/台(X), 龜(O)/亀(X), 佛(O)/仏(X), 國(O)/国(X), 關(O)/関(X), 舊(O)/旧(X), 藝(O)/芸(X), 圖(O)/図(X)
   - 유사 자형 오독 주의: 開/圓, 螭/龜, 奉/春, 栗/栢, 蒙/豪.

6. 고유명사·사찰명·미술사 용어 복원:
   - 인명: 長尾欽彌, 藤島亥治郞 등 당대 관련자.
   - 지명: 慶州郡, 內東面 馬洞里, 陽北面 凡谷里 등 조선 행정구역.
   - 불교/미술사 용어: 阿彌陀如來, 毘盧舍那佛, 舍利塔, 石窟庵, 多寶塔, 釋迦塔, 白雲橋, 青雲橋, 七寶橋, 蓮華橋.
   - 사찰명은 실제 존재한 이름만: 佛國寺, 石窟庵, 芬皇寺, 鳳停寺, 開心寺, 淨惠寺, 栢栗寺.

7. 환각(Hallucination) 방지 — 엄수:
   - 원본에 없는 내용을 생성하지 말 것.
   - 한 글자라도 이미지에서 확신할 수 없으면 해당 글자 뒤에 '[?]' 표기.
   - 완전히 판독 불가능한 구간은 '□' 또는 '(판독불가)'로 표기.

8. 인장·직인·부기 처리:
   - 붉은 인장이 찍힌 위치에는 '(鈐印)' 또는 '(직인: 내용)' 형태로 표기.
   - 관인 테두리로 인한 무의미한 영문자(DES, DID 등)는 삭제.

9. 문서 레이아웃 복원:
   - 세로쓰기(우→좌) 원칙. vision_raw의 배열은 참고만 하고 이미지의 논리적 흐름에 따라 재구성.
   - 공문서 양식은 원본 서식 그대로 유지.

10. 출력 형식 통제:
    - 부가적인 설명, 주석, 마크다운 코드 블록 절대 금지 — 교정된 순수 텍스트만 반환.
    - 빈 줄은 원본 문서의 단락 구분을 따라 유지.
"""

print(f"SYSTEM_PROMPT 길이: {len(SYSTEM_PROMPT):,}자")

## 6) 비용 추적 헬퍼

호출별 토큰·비용을 누적해 마지막에 *총 사용량·청구 금액 예상*을 출력.

In [ ]:
# 모델별 단가 (Gemini API 공식 가격, 2026년 시점)
PRICING_USD_PER_1M = {
    'models/gemini-2.5-flash':      {'input': 0.30, 'output': 2.50},
    'models/gemini-2.5-flash-lite': {'input': 0.10, 'output': 0.40},
    'models/gemini-2.5-pro':        {'input': 1.25, 'output': 10.00},
}

# 세션 누적
session = {'calls': 0, 'input_tokens': 0, 'output_tokens': 0, 'cost_usd': 0.0}


def add_cost(in_tok, out_tok):
    """단일 호출의 토큰·비용을 누적."""
    pricing = PRICING_USD_PER_1M.get(MODEL_NAME, {'input': 0.0, 'output': 0.0})
    cost = in_tok * pricing['input'] / 1_000_000 + out_tok * pricing['output'] / 1_000_000
    session['calls']         += 1
    session['input_tokens']  += in_tok
    session['output_tokens'] += out_tok
    session['cost_usd']      += cost
    return cost

## 7) 일괄 처리 로직

**핵심:**
- Step 1의 JSON을 읽어 `vision_raw`와 원본 이미지를 *함께* Gemini에 전송 (멀티모달).
- 결과는 같은 JSON의 `gemini_corrected` 필드와 별도 `_gemini_final.txt`에 저장.
- 멱등성: 이미 교정된 파일은 건너뜀.
- 429(분당 한도 초과) 발생 시 60초 대기.

In [ ]:
import json
import time
from pathlib import Path
import PIL.Image
from datetime import datetime

INPUT_DIR  = Path(INPUT_DIR)
OUTPUT_DIR = Path(OUTPUT_DIR)

# Step 1 결과 JSON 수집 (.err.json 제외)
json_files = sorted([
    f for f in OUTPUT_DIR.rglob('*.json')
    if not f.name.endswith('.err.json')
])
print(f"교정 대상: 총 {len(json_files)}장")

processed = skipped = errors = 0

for i, json_path in enumerate(json_files, 1):
    try:
        data = json.loads(json_path.read_text(encoding='utf-8'))
    except Exception as e:
        print(f"  [SKIP] {json_path.name} — JSON 읽기 실패: {e}")
        continue

    file_id = json_path.stem
    txt_path = json_path.with_name(f"{file_id}_gemini_final.txt")

    # 멱등성: 이미 교정된 결과가 있으면 스킵
    if data.get('gemini_corrected', '').strip():
        skipped += 1
        continue

    # vision_raw 추출
    vision_raw = data.get('vision_raw', '')
    rel_file_path = data.get('file_path', '')
    img_path = INPUT_DIR / rel_file_path
    if not img_path.exists():
        print(f"  [SKIP] {file_id} — 원본 이미지 없음: {img_path}")
        continue

    print(f"[{i}/{len(json_files)}] {file_id}", flush=True)

    # 프롬프트 구성: SYSTEM_PROMPT + 1차 OCR 텍스트
    combined_prompt = f"{SYSTEM_PROMPT}\n\n[1차 원시 텍스트 (vision_raw)]\n{vision_raw}"
    img = PIL.Image.open(img_path)

    try:
        t0 = time.time()
        response = model.generate_content([combined_prompt, img])
        duration = time.time() - t0
        corrected_text = response.text.strip()

        # JSON 갱신 + TXT 별도 저장
        data['gemini_corrected'] = corrected_text
        data['gemini_corrected_at'] = datetime.now().isoformat(timespec='seconds')
        json_path.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding='utf-8')
        txt_path.write_text(corrected_text, encoding='utf-8')

        # 비용 로깅
        usage = getattr(response, 'usage_metadata', None)
        if usage:
            in_tok  = getattr(usage, 'prompt_token_count', 0) or 0
            out_tok = getattr(usage, 'candidates_token_count', 0) or 0
            cost = add_cost(in_tok, out_tok)
            krw = cost * USD_TO_KRW
            print(f"   -> {len(corrected_text)}자 | {in_tok}+{out_tok} 토큰 | ${cost:.4f} (≈₩{krw:.1f}) | {duration:.1f}s")
        else:
            print(f"   -> {len(corrected_text)}자 | {duration:.1f}s")

        processed += 1
        time.sleep(SLEEP_INTERVAL_SEC)

    except Exception as e:
        msg = str(e)
        if '429' in msg:
            print(f"   -> 할당량 초과(429). 60초 대기 후 다음 파일.")
            time.sleep(60)
        else:
            print(f"   -> 오류: {msg[:120]}")
        errors += 1

print(f"\n완료. 처리 {processed} / 스킵 {skipped} / 오류 {errors}")

## 8) 세션 비용 요약

In [ ]:
if session['calls'] > 0:
    total_krw = session['cost_usd'] * USD_TO_KRW
    avg_krw = total_krw / session['calls']
    print("=" * 50)
    print(f"본 세션 누적: {session['calls']}호출")
    print(f"  입력 토큰:  {session['input_tokens']:>10,}")
    print(f"  출력 토큰:  {session['output_tokens']:>10,}")
    print(f"  총 비용:    ${session['cost_usd']:.4f}  (≈₩{total_krw:,.0f})")
    print(f"  장당 평균:  ≈₩{avg_krw:.1f}")
    print("=" * 50)
else:
    print("이번 세션에서 새로 처리된 호출이 없습니다 (모두 스킵).")

## 9) 결과 점검

처음 한 건만 빠르게 보고, 1차/2차 결과를 *직접 대조*.
점검 항목:
- 명백한 오류(글자 오인식, 행 순서)를 교정했나
- 환각(원본 없는 내용 추가) 발생하지 않았나
- 인장·도장 표기가 일관적인가(`(鈐印)`, `(직인)`)

In [ ]:
samples = [f for f in OUTPUT_DIR.rglob('*.json') if not f.name.endswith('.err.json')]
if samples:
    s = json.loads(samples[0].read_text(encoding='utf-8'))
    print(f"=== {samples[0].stem} ===")
    print(f"\n[1차 Vision] ({len(s.get('vision_raw',''))}자)")
    print(s.get('vision_raw', '')[:400])
    print(f"\n[2차 Gemini] ({len(s.get('gemini_corrected',''))}자)")
    print(s.get('gemini_corrected', '')[:400])

## 10) 다음 단계

이로써 사료 한 장당 다음이 모두 한 JSON에 들어 있다:
- `vision_raw` (1차 OCR)
- `vision_full` (좌표·신뢰도 — 디지털 판본 단계용)
- `gemini_corrected` (2차 문맥 교정)
- 메타데이터 (처리 시각·파일 경로)

별도 `_gemini_final.txt`는 *사람이 직접 읽고 검토하는 용도*.
연구자 판독본(`*_연구자.txt`)을 같은 폴더에 두면, 정답지 비교·정확도 측정으로 이어진다.

**다음으로 가능한 작업:**
- 결과 점검 후 *직접 검토·수정* (매뉴얼 4.3 인용 가이드 참조)
- 정답지 작성 → 정확도 측정 (매뉴얼 4.1)
- 본문 키워드 검색·통계 분석 (매뉴얼 4.2)